## Imports

In [1]:
# Vector processing
import geopandas as gpd
import shapely
import osgb
import pyproj

# Raster processing
import rasterio
import rasterio.features
from skimage import measure
import xarray as xr
import rioxarray

# Visualisation
import matplotlib as mpl
from matplotlib import pyplot as plt
import contextily as cx
import seaborn as sns

%matplotlib widget
mpl.rcParams['axes.formatter.limits'] = (0,0)
FIGSIZE = (10, 7)

# Practical
import os
from shlex import quote
import subprocess
import warnings
from pystac_client import Client as PyStackClient
import boto3
from glob import glob
import functools
import sys
from dask.distributed import Client as DaskClient
from dask.distributed import print as distributed_print

# Calcs and modelling
import numpy as np
import pandas as pd
sys.path.append('../../../../git_packages/PhenoloPy/scripts')
import phenolopy
from scipy.stats import pearsonr

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

WORKING_CRS = 'EPSG:3035'

## Methods

### Biodiversity Data

In [43]:
locations = pd.read_csv('../../../data/Agric_data/Plant/data/locations_2015to2022.csv').set_index('location_id')
locations

,location_type,samplecount
location_id,,
41159,Linear plot,2
41160,Linear plot,2
41163,"Square plot 5 x 5 m, most habitats",2
41164,"Square plot 5 x 5 m, most habitats",2
41165,"Square plot 5 x 5 m, most habitats",2
...,...,...
303537,"Square plot 5 x 5 m, most habitats",1
303538,"Square plot 5 x 5 m, most habitats",1
303550,"Square plot 5 x 5 m, most habitats",1


In [88]:
_temp = pd.read_csv('../../../data/Agric_data/Plant/data/sampleinfowithlatlong_2015to2022.csv', 
                    index_col=0,
                   dtype={'location_id':'Int64'})
sample_info = gpd.GeoDataFrame(_temp,
                                geometry=gpd.GeoSeries.from_xy(_temp['LONGITUDE'],
                                                              _temp['LATITUDE']),
                               crs='EPSG:4326').set_index('id')
sample_info = sample_info.to_crs(WORKING_CRS)
sample_info['date_start'] = pd.to_datetime(sample_info['date_start'])
sample_info['year'] = sample_info['date_start'].dt.year
sample_info

,survey_id,date_start,location_id,entered_sref,entered_sref_system,created_on,created_by_id,LATITUDE,LONGITUDE,monad,monadCRS,country,geometry,year
id,,,,,,,,,,,,,,
19727875,87,2021-07-09,222035,H0851653089,OSIE,2022-07-29 17:29:37,169848,54.426207,-7.868770,H0853,OSIE,Northern_Ireland,POINT (-7.86877 54.42621),2021
19726403,87,2021-07-09,222034,H0863353204,OSIE,2022-07-29 16:44:06,169848,54.427238,-7.866964,H0853,OSIE,Northern_Ireland,POINT (-7.86696 54.42724),2021
6973078,87,2019-09-16,222035,H0851653089,OSIE,2019-10-29 14:56:26,169848,54.426207,-7.868770,H0853,OSIE,Northern_Ireland,POINT (-7.86877 54.42621),2019
6959657,87,2019-08-26,221971,J3583127831,OSIE,2019-10-26 21:33:38,228514,54.181371,-5.919232,J3527,OSIE,Northern_Ireland,POINT (-5.91923 54.18137),2019
6958813,87,2019-08-26,221939,J3533127831,OSIE,2019-10-26 14:11:05,228514,54.181503,-5.926886,J3527,OSIE,Northern_Ireland,POINT (-5.92689 54.18150),2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4026153,154,2018-09-18,203357,TR3758264170,OSGB,2018-09-19 22:34:54,172382,51.326419,1.410581,TR3764,OSGB,Britain,POINT (1.41058 51.32642),2018
1212790,154,2015-08-16,146035,TV5820095500,OSGB,2015-10-05 15:26:19,89944,50.736969,0.242221,TV5895,OSGB,Britain,POINT (0.24222 50.73697),2015
1210754,154,2015-08-16,146018,TV5848495493,OSGB,2015-10-05 10:00:25,89944,50.736829,0.246240,TV5895,OSGB,Britain,POINT (0.24624 50.73683),2015


In [63]:
sample_attributes = pd.read_csv('../../../data/Agric_data/Plant/data/sampleattributes_2015to2022.csv').set_index('sample_attribute_id')
sample_attributes

,sample_id,caption,text_value
sample_attribute_id,,,
3211938,884186,NPMS Survey 1 ID,0
3211939,884186,NPMS Habitat,Neutral pastures and meadows
3211940,884186,Management,Cutting / mowing
3211941,884186,Other Management,mown for recretion
3211942,884186,NPMS Grazers,none
...,...,...,...
69646188,21179861,Rock/gravel/scree,1. < 1% (1-2 indivs)
69646189,21179861,Leaves/litter,4. 5-10%
69646190,21179861,Lichens,8. 51-75%


In [64]:
arable_samples = sample_attributes[sample_attributes['text_value'].str.contains('Arable').fillna(False)]
arable_samples

,sample_id,caption,text_value
sample_attribute_id,,,
3376134,928802,NPMS Habitat,Arable field margins
3376135,928802,Management,Arable cropping
3384293,931623,NPMS Habitat,Arable field margins
3384294,931623,Management,Arable cropping
3384984,931906,NPMS Habitat,Arable field margins
...,...,...,...
68844028,20993210,Management,Arable cropping
68844226,20993259,NPMS Habitat,Arable field margins
68844227,20993259,Management,Arable cropping


In [82]:
arable_samples_loc_yr_unique = sample_info.loc[arable_samples['sample_id'].unique()][['geometry', 'year']].drop_duplicates()
arable_samples_loc_yr_unique

,geometry,year
id,,
928802,POINT (3605072.869 3256121.046),2015
931623,POINT (3310652.883 3197122.957),2015
931906,POINT (3613310.248 3353483.205),2015
932789,POINT (3568858.475 3526359.419),2015
951869,POINT (3436618.076 2977607.276),2015
...,...,...
20850405,POINT (3504923.997 3616610.732),2022
20856232,POINT (3649578.992 3190670.397),2022
20992893,POINT (3516340.240 3296878.081),2022


In [106]:
len(arable_samples_loc_yr_unique[(arable_samples_loc_yr_unique['year'] < 2020) & (arable_samples_loc_yr_unique['year'] > 2017)])

311

In [83]:
buffer_radius = 100
sample_info['study_zone'] = sample_info.geometry.buffer(buffer_radius)
sample_info['study_zone_bbox'] = shapely.box(*np.split(sample_info['study_zone'].bounds.values, 4, axis=1)).flatten()



In [84]:
def create_zone_dir(dir_name):
    '''Builds local directory structure to download data into.'''
    base_path = '../../../data/Agric_data/NPMS_Locations/'
    to_make = os.path.join(base_path, dir_name)
    if os.path.exists(to_make):
        return to_make
    else:
        os.mkdir(to_make)
        distributed_print('Created data dir at ' + to_make)
        return to_make

In [96]:
CROME_layer_refs = {

    '2018': '7cf93df9-2daf-49d7-b4fd-ca8c7126ac73',
    '2019': 'cf4b59e6-693a-49d3-b992-0d75f6b29e40'
}
https://environment.data.gov.uk/backend/catalog/api/geospatial/query?layer=a82c7a21-a7dc-4717-afef-03b5502b5ad1&layer=8dfba8d2-9623-4688-8647-0cf5f62366cd&layer=0c3125b4-0dde-4a11-903d-0f24a7bedf04&layer=a503c875-4ce6-4617-9662-11312a90cea4&layer=b3d7cd5b-06d2-45b7-a965-120b5d0ad702&layer=6ea58d45-21a8-4f0e-8cab-19773ac922f4&layer=62847c64-52a2-428f-bc16-1dd7dc9e7563&layer=36058b65-ef1e-49c5-b8e0-239bfe5de743&layer=d5faafe1-2819-43d9-a18b-26ff7d4d37f7&layer=5dd2e71e-4b84-4302-a721-2b64c283cde0&layer=b016e3e1-283c-4ecf-abea-2dda03ec8149&layer=e095b9c4-4fb1-4402-8606-37347bff2568&layer=808791b7-529f-439c-804a-3b2b3c28e59d&layer=2e41984b-293f-4288-9d69-4c3a664fc1a6&layer=13fde310-ffe8-4468-8f6c-4d1c3efb0521&layer=ec8b2cc1-4475-489c-b628-4e76ec6c7985&layer=83906bf1-44f6-4ad4-9c9e-55bc99e0dfc3&layer=e2b058a6-7ad1-494c-a647-59efb4cf8389&layer=36526f3c-3c74-47c2-a680-5b17ad57b70f&layer=a3d8845f-22e9-470d-b31d-624502356555&layer=716b2438-8840-4fcf-8345-62b09ef1631e&layer=89aed819-d4eb-4866-9b69-6cbb0f7fb76f&layer=edf3fd25-adf8-49f8-9f83-71b25647a6ab&layer=f4a3d6ad-c8b2-4885-a905-7927ff853711&layer=cc864f18-7a7f-4226-a08b-793b83469253&layer=5a545450-bf8c-4097-b71a-7b88fd493c69&layer=297ec657-c859-42dc-a611-841a5c625560&layer=20a21e2f-be47-4157-b42f-9638a8d23ff5&layer=8b2c07ce-e91e-4374-b038-9acdf9fe5ffd&layer=862a3576-a1b7-482b-8c33-9df53254ec33&layer=e762f438-f330-4b20-8c43-9e70c0644b41&layer=b1bc060d-3978-4012-8621-a01cc96fdc39&layer=8b3401c8-98e9-4736-be40-ac56eaf52859&layer=3fd18a98-a42a-473b-8671-b2877d44a14b&layer=084b7afa-e846-4211-a213-b4d0efb5d5db&layer=175c9629-b69a-4274-8103-bddd5b262553&layer=81c836a9-158e-41b8-89c7-c6a2c35f9ec1&layer=2d14914b-0958-4741-b0f1-c7287780f180&layer=f2c71f86-442d-4079-80d1-ca6742d40695&layer=27cadec6-760e-4c19-ac7d-ce191091b814&layer=9d59d402-f4f8-4069-bc6d-d805828af7f2&layer=2d3fc4bc-6eca-4bf0-90cb-de03e9d5430a&layer=7faf2839-27ff-43a4-a433-e036cedf3f90&layer=7e31d605-cda2-4ccf-801a-e79e17ab2573&layer=ad46a571-fb1e-41f2-87c7-eb0c0f24ea73&layer=36399584-1757-4c80-bfb4-ec94f50cb3c3&layer=e2acdafb-623a-40ff-b62f-f0aa66054035

In [95]:
# For each location and year:
# Note that a given location ID can be associated with multiple geometries (ie because
# volunteers sample multiple sampling zones within one "location")
sample_ref = arable_samples_loc_yr_unique.iloc[0].name
year = sample_info.loc[sample_ref]['year']

# Get lat lon, buffer and define bbox
transform_working_to_geodetic = pyproj.Transformer.from_crs(pyproj.CRS(WORKING_CRS), 
                                                            pyproj.CRS('EPSG:4326'), 
                                                            always_xy=True).transform

aoi_bbox = sample_info.loc[sample_ref]['study_zone_bbox'] # in projected coords
aoi_bbox_geodetic = transform_working_to_geodetic(*aoi_bbox.exterior.xy)

# Send that as a request to the correct CROME dataset

dir_name = 'location_' + str(sample_ref)

data_path = create_zone_dir(dir_name)
crome_dir = os.path.join(data_path, 'CROME')
subprocess.run('../../../data/Agric_data/get_crome.sh ' + \
                       '\"' + crome_dir + '\" ' + \
                       '\'' + f'{{"coordinates": {np.array2string(np.dstack(aoi_bbox_geodetic), precision=8, separator=",", suppress_small=True)},"type":"Polygon"}}'.replace("\n", "") + '\'',
           shell=True)

# If the immediately adjacent pixels are spring cereals and this sample is on agricultural
# land instead of at boundary or adjacent, do not delete

(44216, 2015)